In [8]:
def get_predictions(model, data, batch_size=64):
    """ Get predictions on data for a classification model m returning predictions and true labels"""
    model.to("cpu")
    model.eval()
 
    all_predictions = []
    all_confidences = []
    
    if len(data['input_ids']) % batch_size != 0:
        num_batches = len(data['input_ids']) // batch_size + 1
    else:
        num_batches = len(data['input_ids']) // batch_size

    for i in tqdm(range(num_batches),desc= "Prediction Progress"):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(data['input_ids'])) # prevent going out of bount input_ids has same size as attention mask should
        input_ids_batch = torch.tensor(data['input_ids'][start_idx:end_idx])
        attention_mask_batch = torch.tensor(data['attention_mask'][start_idx:end_idx])
        with torch.no_grad():  # Disable gradient computation for inference
            predictions = model(input_ids_batch, attention_mask=attention_mask_batch)
            logits = predictions.logits
            predictions = torch.argmax(logits, dim=-1) # make sure it is a ser
            probs = torch.nn.functional.softmax(logits, dim=-1) # return a proabiblity distribution
            max_probs, _ = torch.max(probs, dim=-1)
            
        all_confidences.append(max_probs)  # this tells us how confident we are with an value
        all_predictions.append(predictions)
        # print(predictions)
        # print("--------")
    # print(all_predictions)
    
    
    #Concatenate all batch predictions into one tensor since we currently have a list of lists
    all_predictions = torch.cat(all_predictions, dim=0)
    all_confidences = torch.cat(all_confidences, dim=0)
    
    return all_predictions , all_confidences



def self_training_loop(model, train_dataset, tokenizer, confidence_threshold=0.9, num_epochs=1, mode='unlabeled',threshold_mode = 'fix'):
    
    # still need to better develop still bad code since exit really randomly depend on the confidence_threshold which is not good for big data waiting for the final data point to set in
    
    
    data_collator = DataCollatorWithPadding(tokenizer) 
    training_args = TrainingArguments(
        num_train_epochs=num_epochs,      # Number of training epochs
    )

    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,    # Training dataset   
        data_collator=data_collator,
    )
    
    #https://github.com/tqdm/tqdm#usage
    unlabeled_mask =  train_dataset["label"] == -1
    total_unlabeled = sum(label == -1 for label in train_dataset["label"])
    pbar = tqdm(total=total_unlabeled, desc="Unlabeled Samples Remaining")

    labeled_train_dataset = train_dataset.filter(lambda x: x["label"] != -1)
    trainer.train_dataset = labeled_train_dataset  # Update trainer with labeled dataset
    print(f"data: { labeled_train_dataset =}")
    # https://peps.python.org/pep-0289/
    print(sum(label == -1 for label in labeled_train_dataset["label"]))
    print("hello")
    trainer.train()

    while -1 in train_dataset["label"]:
        # trainer.train()
        predictions, confidences = get_predictions(model, train_dataset)
        
        if threshold_mode == 'disable':
            threshold_value = 0
        elif threshold_mode == 'fix':
            threshold_value = confidence_threshold
        elif threshold_mode == 'quantile':
            # this mode with too higih quantile does have converge issue
            threshold_value = torch.quantile(confidences, confidence_threshold).item()
        else:
            print("threshold mode not specified")

        print(confidences)
        print(predictions)
        print(len(train_dataset))
        print("------------------------")
        if mode == 'unlabeled':
        # we might want to make distinction here is the train_dataset label only or etc
        #  It allows you to apply a processing function to each example in a dataset, independently or in batches. This function can even create new rows and columns.
        # https://huggingface.co/docs/datasets/en/process
            condition_met_indices = [] # this is just here to keep track of the pesudo labeled added per iteration if needed
            # example here is each row
            def update_labels(example, idx):
                # Update labels for unlabeled data only, if confidence is above the threshold
                if example['label'] == -1 and confidences[idx] >= threshold_value:
                    example['label'] = predictions[idx]
                    condition_met_indices.append(idx)
                return example

            # Use .map() to apply the updates to all examples where the condition is met
            train_dataset = train_dataset.map(update_labels, with_indices=True)
            print(f"Indices where the condition was met: {condition_met_indices}")
                  

        # untested yet !!!!!!!
        elif mode == "all":
            def update_labels_all(example, idx):
                if confidences[idx] >= threshold_value:
                    example['label'] = predictions[idx]
                    condition_met_indices.append(idx)
                return example

            train_dataset = train_dataset.map(update_labels_all, with_indices=True)
            print(f"Indices where the condition was met: {condition_met_indices}")
                    
        # Update trainer with the latest training data
        pbar.update(len(condition_met_indices))
        trainer.train_dataset = train_dataset
            
        if -1 not in train_dataset["label"]:
          print("All labels are filled. Stopping training.")
          break
        
    return model, train_dataset

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset
from trl import IterativeSFTTrainer
from tqdm import tqdm




# We are still in datafre

review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv")
# review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv",nrows=100)

print(review_data["review_comment"].isna().sum())
review_data["review_comment"] = review_data["review_comment"].fillna("") 
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})



# vectorizer = TfidfVectorizer()

# review_X = vectorizer.fit_transform(review_data["review_comment"])  # Convert text to TF-IDF features

# # Array representing the labels. Unlabeled samples should have the label -1.

review_data["recommend"] = review_data["recommend"].fillna(-1)


# print("wow")
# print(review_data[review_data["recommend"]==-1])


# for some reason the tyepe matter https://discuss.huggingface.co/t/valueerror-target-size-torch-size-8-must-be-the-same-as-input-size-torch-size-8-8/12133/9
train_df = pd.DataFrame({"text": review_data["review_comment"], "label": review_data["recommend"].astype(int)})



# Now we are working with huggingface daset 

train_dataset = Dataset.from_pandas(train_df)

print(sum(label == -1 for label in train_dataset["label"]))
print(train_dataset["label"])
unlabeled_mask =  train_dataset["label"] == -1
# print(unlabeled_mask)
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Bert model accepts max 512
# Tokenize datasets
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding='max_length', max_length=512)

#------------------------------------


#------------------------------------

# In Hugging Face models like BERT, the features come from the tokenized input data that is fed into the model.
# the map applie the tokenize function to each batch of the function 
train_dataset = train_dataset.map(tokenize, batched=True)

# print(train_dataset)
# Load model
# define model



# print (f"we are using {device}")

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
device = torch.device("cpu")
model.to(device)


model, train_dataset = self_training_loop(model, train_dataset, tokenizer, threshold_mode = 'disable')

# Save the trained model and tokenizer
model.save_pretrained("self_trained_bert_base_model")
tokenizer.save_pretrained("self_trained_base_tokenizer")

# Save the final labeled dataset
train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
train_dataset_df = train_dataset_df[["label","text"]]
train_dataset_df.to_csv("final_labeled_reviews.csv", index=False)


#---------Evaluate-----------



train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
train_dataset_df = train_dataset_df[["label","text"]]
train_dataset_df.to_csv("final_labeled_reviews.csv", index=False)
review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv")
# review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv",nrows=100)



print(review_data_test ["review_comment"].isna().sum())
review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})
review_data_test ["recommend"] = review_data_test ["recommend"].fillna(-1)



review_y_test = review_data_test["recommend"]

train_dataset_frame =  train_dataset.to_pandas()
# only the unlabled data
mask = review_data["recommend"] == -1 
review_y_test =  review_y_test[mask]

final_data_set = train_dataset_frame[mask]
# print(review_y_test)
# print(final_data_set)

# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test, final_data_set["label"] )
precision = precision_score(review_y_test, final_data_set["label"]  ,average='macro')
recall = recall_score(review_y_test,final_data_set["label"]  ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")

with open('evaluation_results_selftrain_noboot_full_bert_models.txt', 'w') as f:
    f.write(f"Accuracy after fine-tuning: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")



1
25056
[-1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1, -1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, -1, 1, 1, 0, 1, 1, -1, 1, 0, 1, -1, 1, -1, 1, -1, 0, 1, -1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, 1, -1, 1, -1, 1, -1, 0, -1, 1, -1, 1, -1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 0, 1, 1, -1, -1, 1, 1, 1, -1, 1, 1, 1, -1, 1, -1, -1, -1, 1, -1, 1, 1, 1, 1, -1, 1, -1, 0, -1, 1, -1, 0, 1, 0, -1, -1, -1, 1, 1, -1, 1, 1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, -1, 1, 1, 1, 1, 1, 1, -1, 1, 1, -1, 1, -1, 1, 1, -1, 1, -1, -1, 1, 1, 1, -1, -1, 1, -1, 1, 1, 1, 1, -1, -1, 1, 1, 1, -1, 1, 1, 1, 1, -1, 0, 1, 1, 1, 1, 1, -1, 1, 1, 0, 1, 1, -1, 1, 1, -1, 1, -1, 1, 1, -1, 1, -1, 1, -1, 1, 1, 1, -1, 1, -1, 1, -1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, -1, -1, -1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1, -1, -1, -1, -1, -1, 1, -1, 1, -1, -1, 1, 1, 1, 1, 1, -1, -1, -1, -1, -1, 1, 1, -1, 1, -1, -1, 1, 1, 1, 1, -1

Map:   0%|          | 0/83522 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Filter:   0%|          | 0/83522 [00:00<?, ? examples/s]

data:  labeled_train_dataset =Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 58466
})
0
hello


...

tensor([0.9886, 0.9938, 0.9982,  ..., 0.9918, 0.9858, 0.9913])
tensor([1, 1, 1,  ..., 1, 1, 1])
83522
------------------------


Map:   0%|          | 0/83522 [00:00<?, ? examples/s]

Unlabeled Samples Remaining: 100%|██████████| 25056/25056 [17:25:59<00:00,  2.50s/it]

Indices where the condition was met: [0, 9, 13, 17, 18, 26, 39, 42, 48, 52, 54, 56, 59, 67, 68, 72, 74, 76, 78, 80, 82, 88, 90, 95, 96, 100, 104, 106, 107, 108, 110, 115, 117, 119, 121, 125, 126, 127, 130, 138, 140, 143, 150, 153, 155, 158, 160, 161, 165, 166, 168, 173, 174, 178, 183, 190, 196, 199, 201, 204, 206, 208, 212, 214, 216, 217, 243, 244, 245, 246, 251, 257, 265, 270, 271, 272, 273, 274, 276, 278, 279, 285, 286, 287, 288, 289, 292, 294, 295, 300, 302, 307, 310, 311, 315, 316, 317, 324, 325, 327, 329, 330, 340, 345, 346, 350, 352, 354, 358, 362, 363, 367, 369, 374, 378, 388, 391, 397, 398, 403, 408, 415, 417, 420, 425, 428, 433, 436, 437, 438, 439, 444, 446, 448, 449, 451, 457, 463, 467, 472, 474, 475, 481, 482, 497, 499, 508, 520, 521, 524, 533, 535, 540, 543, 545, 546, 548, 549, 552, 554, 557, 558, 563, 566, 567, 568, 569, 583, 584, 585, 588, 593, 594, 595, 600, 602, 606, 607, 611, 614, 615, 616, 617, 627, 628, 629, 632, 633, 634, 640, 643, 644, 646, 648, 653, 656, 657, 661,

1

Evaluation Results on Test Data:
Accuracy:  0.899345466155811
Precision: 0.889972039129923
Recall:    0.8795824523434054


In [ ]:
print(Original ["review_comment"].isna().sum())
Original ["review_comment"] = Original ["review_comment"].fillna("") 
Original ["recommend"] = Original ["recommend"].map({"Recommended": 1, "Not Recommended": 0})




review_y_test = Original["recommend"]


print(review_y_test )

# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled       data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.
accuracy = accuracy_score(review_y_test, Baseline_distiltbert_prediction["label"] )
precision = precision_score(review_y_test, Baseline_distiltbert_prediction["label"]  ,average='macro')
recall = recall_score(review_y_test,Baseline_distiltbert_prediction["label"]  ,average='macro')

print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")
